<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/VITPyTorchPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,batch_first=True)

    self.layerNorm1 = nn.LayerNorm(embed_dim)# here embed_dim is just for the input shape, we can still set espilon

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.2)
    )

    self.layerNorm2 = nn.LayerNorm(embed_dim)# here embed_dim is just for the input shape, we can still set espilon

  def forward(self,input):
    att,_ = self.attention(input,input,input) # here another _ variable is for the weights
    att = self.layerNorm1(input + att)
    mlp_output = self.mlp(att)
    return self.layerNorm2(att + mlp_output)

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self,embed_dim,patch_size) -> None:
    super().__init__()

    self.patchEmbed = nn.Conv2d(in_channels=3,out_channels=embed_dim,kernel_size=patch_size,stride=patch_size)

  def forward(self,input):
    patchs = self.patchEmbed(input)# (Batchsize,embed_dim,patch_size) -> here patch is still 14x14
    flattenedPatch = patchs.flatten(2) # now patchs are flattened(196)
    flattenedPatch = flattenedPatch.transpose(1,2) # swap index 1 and 2
    return flattenedPatch

In [ ]:
class VisionTransformer(nn.Module):
  def __init__(self,embed_dim,patch_size,image_size,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.patchEmbed = PatchEmbedding(embed_dim=embed_dim,patch_size=patch_size)
    num_patchs = (image_size // patch_size) ** 2
    self.positionEmbed = nn.Parameter(torch.zeros(1,num_patchs,embed_dim)) #here 1 means batch size,num_patchs is range,embed_dim is how many

    self.transform_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])
    self.output_layer = nn.Linear(in_features=embed_dim,out_features=num_classes)

  def forward(self,input):
    patchs = self.patchEmbed(input)
    x = patchs + self.positionEmbed

    for transform_layer in self.transform_layers:
      x = transform_layer(x)

    x = x.mean(dim=1)# here dim 1 means from (batchsize,patchs,embed_dim) take patchs and get average of that

    return self.output_layer(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:
model = VisionTransformer(embed_dim=256,
                          patch_size=14,
                          image_size=224,
                          num_heads=4,
                          ff_dim=128,
                          num_layers=8,
                          num_classes=4).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(),lr=0.0001)

In [ ]:
epochs =10

for epoch in range(epochs):
  model.train()
  trian_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.max(output,1) # here 1 means
    train_correct += (pred == labels).sum.item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in val_loader:
     images = images.to(device)
     labels = labels.to(device)


     output = model(images)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     pred = torch.max(output,1) # here 1 means
     val_correct += (pred == labels).sum.item()
     val_total += labels.size(0)

    val_accuracy = val_correct / val_total